# Notebook 3 - Pipeline Execution with Sample Complaints

Loads the sample complaints, runs the full classify -> analyze -> gate -> route pipeline per complaint, and persists structured outputs. Runs locally on the mock backend; swap in `VLLMClient` on the AMD cloud with no other changes.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from complaint_router.pipeline import run_pipeline
from complaint_router.llm.mock_client import MockClient
from complaint_router.schemas import ComplaintInput

DATA = Path('..') / 'data' / 'sample_complaints.json'
rows = json.loads(DATA.read_text(encoding='utf-8'))
client = MockClient()  # cloud: from complaint_router.llm.vllm_client import VLLMClient; client = VLLMClient()
len(rows)

In [ ]:
records = []
for r in rows:
    complaint = ComplaintInput(**{k: r[k] for k in ('complaint_id', 'channel', 'complaint_text', 'received_at')})
    res = run_pipeline(complaint, client)
    records.append({
        'complaint_id': res.complaint_id,
        'category': res.classification.category.value if res.classification else None,
        'severity': res.analysis.severity_level.value if res.analysis else None,
        'intention': res.analysis.intention.value if res.analysis else None,
        'final_confidence': res.gate.final_confidence if res.gate else None,
        'route_status': res.route.route_status.value,
        'destination_team': res.route.destination_team,
        'priority_hint': res.route.priority_hint.value,
        'routing_note': res.route.routing_note,
        'gold_category': r.get('gold_category'),
    })
df = pd.DataFrame(records)
df

In [ ]:
out = Path('..') / 'data' / 'pipeline_outputs.json'
df.to_json(out, orient='records', indent=2)
print('wrote', out)

## Try a custom complaint (text only)

Edit `my_complaint` below and re-run the cell. Only the complaint text is
needed — the id, channel, and timestamp are filled in automatically. Uses the
same `client` defined above (mock locally, vLLM on the cloud).

In [ ]:
# Self-contained: re-imports so this cell works even if run before the others.
from complaint_router.pipeline import run_pipeline
from complaint_router.llm.mock_client import MockClient
from complaint_router.schemas import ComplaintInput, utcnow

# Reuse the `client` from the setup cell if present, else make a mock one here.
client = globals().get("client") or MockClient()


def route_text(text, llm=client):
    """Run the full pipeline on raw complaint text and show every stage."""
    complaint = ComplaintInput(
        complaint_id="MANUAL",
        channel="manual",
        complaint_text=text,
        received_at=utcnow(),
    )
    res = run_pipeline(complaint, llm)
    if res.classification:
        print(f"[1] category   : {res.classification.category.value} "
              f"(conf {res.classification.category_confidence})")
    if res.analysis:
        print(f"[2] severity   : {res.analysis.severity_level.value}  "
              f"intention: {res.analysis.intention.value} "
              f"(conf {res.analysis.analysis_confidence})")
    if res.gate:
        print(f"[3] final_conf : {res.gate.final_confidence:.2f}  "
              f"auto_route={res.gate.auto_route} ({res.gate.escalation_reason})")
    print(f"[4] route      : {res.route.route_status.value} -> "
          f"{res.route.destination_team} (priority {res.route.priority_hint.value})")
    return res


# Edit this line and re-run:
my_complaint = "I was charged twice for my booking and need a refund urgently"
_ = route_text(my_complaint)